# TCAV Concept-Based Explanations for NeuroState

Tests whether NeuroState's internal representations align with
clinically meaningful EEG concepts:
   Sleep-EDF: delta waves (N3), sleep spindles (N2), alpha rhythm (Wake)
   CHB-MIT: high-frequency activity (seizure), amplitude increase (seizure)

Method:
   1. Define concepts via frequency band power in specific ranges
   2. Extract activations from transformer bottleneck layer
   3. Train linear concept classifiers (CAVs)
   4. Compute TCAV scores: does the model's prediction change in the
      direction of the concept?

This is the first application of TCAV to an EEG boundary detection
transformer, a novel interpretability contribution.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import h5py
import json
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from scipy.signal import welch
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

figures_dir = Path("figures/tcav")
figures_dir.mkdir(parents=True, exist_ok=True)

Device: cuda


In [2]:
# Publication-quality figure settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 13,
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.titlesize': 16,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.15,
})

In [3]:
# Define EEG concepts via frequency band power

def compute_band_powers(epoch_data, sfreq=100):
    """
    Compute frequency band powers for a single epoch.
    epoch_data: (n_channels, n_samples)
    Returns dict of band powers averaged across channels.
    """
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'sigma': (11, 16),  # sleep spindle band
        'beta': (13, 30),
        'gamma': (30, 45),
    }

    powers = {}
    for ch in range(epoch_data.shape[0]):
        freqs, psd = welch(epoch_data[ch], fs=sfreq, nperseg=min(256, epoch_data.shape[1]))
        for band_name, (lo, hi) in bands.items():
            mask = (freqs >= lo) & (freqs <= hi)
            bp = np.trapz(psd[mask], freqs[mask])
            if band_name not in powers:
                powers[band_name] = []
            powers[band_name].append(bp)

    return {k: np.mean(v) for k, v in powers.items()}


def label_concepts_sleep(epochs, labels, sfreq=100):
    """
    Create binary concept labels for Sleep-EDF based on frequency content.

    Concepts:
      - delta_dominant: high delta power (correlates with N3)
      - spindle_present: high sigma power (correlates with N2)
      - alpha_dominant: high alpha power (correlates with Wake/REM)
    """
    print("Computing frequency features for concept labeling...")
    all_powers = []
    for i in tqdm(range(len(epochs)), desc="Band powers"):
        all_powers.append(compute_band_powers(epochs[i], sfreq))

    delta_vals = np.array([p['delta'] for p in all_powers])
    sigma_vals = np.array([p['sigma'] for p in all_powers])
    alpha_vals = np.array([p['alpha'] for p in all_powers])
    total_power = delta_vals + np.array([p['theta'] for p in all_powers]) + \
                  alpha_vals + sigma_vals + np.array([p['beta'] for p in all_powers])

    # Relative powers
    delta_rel = delta_vals / (total_power + 1e-10)
    sigma_rel = sigma_vals / (total_power + 1e-10)
    alpha_rel = alpha_vals / (total_power + 1e-10)

    # Threshold at median for binary concept labels
    concepts = {
        'delta_dominant': (delta_rel > np.percentile(delta_rel, 70)).astype(int),
        'spindle_present': (sigma_rel > np.percentile(sigma_rel, 70)).astype(int),
        'alpha_dominant': (alpha_rel > np.percentile(alpha_rel, 70)).astype(int),
    }

    # Print concept-class alignment
    stage_names = ['Wake', 'N1', 'N2', 'N3', 'REM']
    for cname, clabels in concepts.items():
        print(f"\n  Concept: {cname}")
        for stage in range(5):
            mask = labels == stage
            if mask.any():
                pct = clabels[mask].mean() * 100
                print(f"    {stage_names[stage]}: {pct:.1f}% positive")

    return concepts


def label_concepts_seizure(epochs, labels, sfreq=100):
    """
    Create binary concept labels for CHB-MIT.

    Concepts:
      - high_frequency: elevated beta+gamma (correlates with seizure)
      - high_amplitude: high signal variance (correlates with seizure)
    """
    print("Computing frequency features for seizure concepts...")
    all_powers = []
    all_variance = []
    for i in tqdm(range(len(epochs)), desc="Band powers"):
        all_powers.append(compute_band_powers(epochs[i], sfreq))
        all_variance.append(np.var(epochs[i]))

    beta_gamma = np.array([p['beta'] + p['gamma'] for p in all_powers])
    variance = np.array(all_variance)

    concepts = {
        'high_frequency': (beta_gamma > np.percentile(beta_gamma, 70)).astype(int),
        'high_amplitude': (variance > np.percentile(variance, 70)).astype(int),
    }

    for cname, clabels in concepts.items():
        print(f"\n  Concept: {cname}")
        for cls, cls_name in [(0, 'Normal'), (1, 'Seizure')]:
            mask = labels == cls
            if mask.any():
                pct = clabels[mask].mean() * 100
                print(f"    {cls_name}: {pct:.1f}% positive")

    return concepts

In [4]:
# Extract model activations

def extract_activations(model, epochs, config, device, layer_name='encoder',
                        n_channels=3, max_samples=2000):
    """
    Extract bottleneck activations for each epoch.
    Returns pooled activations: (N, embed_dim)
    """
    # Import model components (same as other notebooks)
    model.eval()
    all_acts = []

    n = min(len(epochs), max_samples)
    indices = np.random.choice(len(epochs), n, replace=False)
    indices.sort()

    for start in tqdm(range(0, n, 32), desc="Extracting activations"):
        batch_idx = indices[start:start+32]
        batch_eps = []
        for idx in batch_idx:
            ep = epochs[idx].astype(np.float32)
            nc = ep.shape[0]
            if nc < n_channels:
                ep = np.vstack([ep, np.zeros((n_channels-nc, ep.shape[1]), dtype=np.float32)])
            elif nc > n_channels:
                ep = ep[:n_channels]
            # Create 3-epoch window (center only matters, pad with same)
            batch_eps.append(np.stack([ep, ep, ep], axis=0))

        x = torch.tensor(np.stack(batch_eps), dtype=torch.float32).to(device)

        with torch.no_grad():
            B, N, C, T = x.shape
            embs = [model.mr_encoder(x[:,i]) + model.epoch_embed[:,i] for i in range(N)]
            full = model.pos_drop(torch.cat(embs, dim=1) + model.pos_embed)

            if layer_name == 'encoder':
                # Use encoder output (before transformer blocks)
                tpe = model.tokens_per_epoch
                pooled = full[:, tpe:2*tpe, :].mean(dim=1)
            else:
                # Use post-transformer output
                cp = model.changepoint_module(full)
                bnd = cp['boundaries']
                for blk in model.blocks:
                    full = blk(full, bnd)
                full = model.norm(full)
                tpe = model.tokens_per_epoch
                pooled = full[:, tpe:2*tpe, :].mean(dim=1)

            all_acts.append(pooled.cpu().numpy())

    return np.concatenate(all_acts, axis=0), indices

In [5]:
# Train CAVs and compute TCAV scores

def train_cav(activations, concept_labels, n_splits=5):
    """
    Train a linear classifier (CAV) to separate concept-positive from
    concept-negative examples in activation space.
    Returns: CAV direction (weight vector), mean accuracy across folds.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accs = []
    all_weights = []

    for train_idx, test_idx in skf.split(activations, concept_labels):
        clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        clf.fit(activations[train_idx], concept_labels[train_idx])
        acc = clf.score(activations[test_idx], concept_labels[test_idx])
        accs.append(acc)
        all_weights.append(clf.coef_[0])

    cav_direction = np.mean(all_weights, axis=0)
    cav_direction = cav_direction / (np.linalg.norm(cav_direction) + 1e-10)

    return cav_direction, np.mean(accs)


def compute_tcav_score(model, epochs, labels, target_class, cav_direction,
                       config, device, n_channels=3, max_samples=500):
    model.eval()
    target_indices = np.where(labels == target_class)[0]
    if len(target_indices) > max_samples:
        target_indices = np.random.choice(target_indices, max_samples, replace=False)

    cav_tensor = torch.tensor(cav_direction, dtype=torch.float32).to(device)
    positive_count = 0
    total_count = 0

    for start in range(0, len(target_indices), 16):
        batch_idx = target_indices[start:start+16]
        batch_eps = []
        for idx in batch_idx:
            ep = epochs[idx].astype(np.float32)
            nc = ep.shape[0]
            if nc < n_channels:
                ep = np.vstack([ep, np.zeros((n_channels-nc, ep.shape[1]), dtype=np.float32)])
            elif nc > n_channels:
                ep = ep[:n_channels]
            batch_eps.append(np.stack([ep, ep, ep], axis=0))

        x = torch.tensor(np.stack(batch_eps), dtype=torch.float32).to(device)

        # Forward through encoder
        B, N, C, T = x.shape
        embs = [model.mr_encoder(x[:,i]) + model.epoch_embed[:,i] for i in range(N)]
        full = torch.cat(embs, dim=1)
        full = model.pos_drop(full + model.pos_embed)

        # This is the bottleneck — keep it in the graph
        tpe = model.tokens_per_epoch
        # Pool center epoch embeddings as our activation layer
        h = full[:, tpe:2*tpe, :].mean(dim=1)  # (B, embed_dim)
        h.requires_grad_(True)
        h.retain_grad()

        # Rebuild full sequence with h injected back
        # Replace center epoch tokens with a version that routes through h
        center_tokens = h.unsqueeze(1).expand(-1, tpe, -1)  # (B, tpe, embed_dim)
        full_modified = torch.cat([
            full[:, :tpe, :].detach(),
            center_tokens,
            full[:, 2*tpe:, :].detach()
        ], dim=1)

        # Continue forward from modified sequence
        cp = model.changepoint_module(full_modified)
        bnd = cp['boundaries']
        seq = full_modified
        for blk in model.blocks:
            seq = blk(seq, bnd)
        seq = model.norm(seq)
        pooled = seq[:, tpe:2*tpe, :].mean(dim=1)
        logits = model.head(pooled)

        target_logit = logits[:, target_class].sum()
        target_logit.backward()

        if h.grad is not None:
            grads = h.grad.detach()
            dot = (grads * cav_tensor.unsqueeze(0)).sum(dim=1)
            positive_count += (dot > 0).sum().item()
            total_count += len(dot)

        model.zero_grad()

    if total_count == 0:
        return 0.5
    return positive_count / total_count

In [6]:
# Run TCAV analysis

def run_tcav_sleep(model, h5_path, device, figures_dir):
    """Full TCAV analysis on Sleep-EDF."""
    print("\n" + "=" * 60)
    print("TCAV ANALYSIS — SLEEP-EDF")
    print("=" * 60)

    with h5py.File(h5_path, 'r') as f:
        epochs = f['epochs'][:]
        labels = f['labels'][:]

    # Label concepts
    concepts = label_concepts_sleep(epochs, labels)

    # Extract activations
    activations, used_indices = extract_activations(
        model, epochs, None, device, layer_name='encoder',
        n_channels=3, max_samples=2000)
    used_labels = labels[used_indices]

    # Train CAVs
    print("\nTraining CAVs:")
    cavs = {}
    for cname, clabels in concepts.items():
        used_clabels = clabels[used_indices]
        direction, acc = train_cav(activations, used_clabels)
        cavs[cname] = direction
        print(f"  {cname}: CAV accuracy = {acc:.3f}")

    # Compute TCAV scores for each class × concept
    stage_names = ['Wake', 'N1', 'N2', 'N3', 'REM']
    print("\nTCAV Scores (>0.5 = concept positively influences prediction):")

    tcav_matrix = np.zeros((5, len(concepts)))
    for c_idx, (cname, cav) in enumerate(cavs.items()):
        for stage in range(5):
            score = compute_tcav_score(
                model, epochs, labels, stage, cav, None, device,
                n_channels=3, max_samples=300)
            tcav_matrix[stage, c_idx] = score
            print(f"  {stage_names[stage]} × {cname}: {score:.3f}")

    # Plot
    fig, ax = plt.subplots(figsize=(9, 6.5))
    im = ax.imshow(tcav_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(concepts)))
    ax.set_xticklabels(list(concepts.keys()), fontsize=13, rotation=25, ha='right')
    ax.set_yticks(range(5))
    ax.set_yticklabels(stage_names, fontsize=14, fontweight='bold')
    for i in range(5):
        for j in range(len(concepts)):
            val = tcav_matrix[i, j]
            text_color = 'white' if val < 0.30 or val > 0.75 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=15, fontweight='bold', color=text_color)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('TCAV Score', fontsize=13, labelpad=8)
    cbar.ax.tick_params(labelsize=11)
    ax.set_title('TCAV: Clinical EEG Concepts × Sleep Stages',
                fontsize=16, fontweight='bold', pad=12)
    ax.set_xlabel('EEG Concept', fontsize=14, labelpad=10)
    ax.set_ylabel('Sleep Stage', fontsize=14, labelpad=8)
    ax.tick_params(axis='both', which='both', length=0)
    plt.tight_layout()
    plt.savefig(figures_dir / 'tcav_sleep.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"\nSaved: {figures_dir / 'tcav_sleep.png'}")

    return tcav_matrix


def run_tcav_seizure(model, h5_path, device, figures_dir):
    """Full TCAV analysis on CHB-MIT."""
    print("\n" + "=" * 60)
    print("TCAV ANALYSIS — CHB-MIT")
    print("=" * 60)

    with h5py.File(h5_path, 'r') as f:
        epochs = f['epochs'][:]
        labels = f['labels'][:]

    concepts = label_concepts_seizure(epochs, labels)

    activations, used_indices = extract_activations(
        model, epochs, None, device, layer_name='encoder',
        n_channels=17, max_samples=2000)
    used_labels = labels[used_indices]

    print("\nTraining CAVs:")
    cavs = {}
    for cname, clabels in concepts.items():
        used_clabels = clabels[used_indices]
        direction, acc = train_cav(activations, used_clabels)
        cavs[cname] = direction
        print(f"  {cname}: CAV accuracy = {acc:.3f}")

    class_names = ['Normal', 'Seizure']
    print("\nTCAV Scores:")
    tcav_matrix = np.zeros((2, len(concepts)))
    for c_idx, (cname, cav) in enumerate(cavs.items()):
        for cls in range(2):
            score = compute_tcav_score(
                model, epochs, labels, cls, cav, None, device,
                n_channels=17, max_samples=300)
            tcav_matrix[cls, c_idx] = score
            print(f"  {class_names[cls]} × {cname}: {score:.3f}")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    im = ax.imshow(tcav_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(concepts)))
    ax.set_xticklabels(list(concepts.keys()), fontsize=14)
    ax.set_yticks(range(2))
    ax.set_yticklabels(class_names, fontsize=14, fontweight='bold')
    for i in range(2):
        for j in range(len(concepts)):
            val = tcav_matrix[i, j]
            text_color = 'white' if val < 0.30 or val > 0.75 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=17, fontweight='bold', color=text_color)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('TCAV Score', fontsize=13, labelpad=8)
    cbar.ax.tick_params(labelsize=11)
    ax.set_title('TCAV: EEG Concepts × Seizure Detection',
                fontsize=16, fontweight='bold', pad=12)
    ax.set_xlabel('EEG Concept', fontsize=14, labelpad=8)
    ax.tick_params(axis='both', which='both', length=0)
    plt.tight_layout()
    plt.savefig(figures_dir / 'tcav_seizure.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"\nSaved: {figures_dir / 'tcav_seizure.png'}")

    return tcav_matrix

In [7]:
# MODEL 

class PatchEmbedding(nn.Module):
    def __init__(self, n_channels=3, n_samples=3000, embed_dim=128, temporal_kernel=25, pool_kernel=75, pool_stride=15, dropout=0.1):
        super().__init__()
        self.temporal_conv = nn.Sequential(nn.Conv2d(1,40,(1,temporal_kernel),padding=(0,temporal_kernel//2)),nn.BatchNorm2d(40),nn.GELU())
        self.spatial_conv = nn.Sequential(nn.Conv2d(40,40,(n_channels,1)),nn.BatchNorm2d(40),nn.GELU())
        self.pool = nn.AvgPool2d((1,pool_kernel),stride=(1,pool_stride))
        self.projection = nn.Sequential(nn.Conv2d(40,embed_dim,(1,1)),nn.Dropout(dropout))
        self.seq_len = (n_samples-pool_kernel)//pool_stride+1
    def forward(self,x):
        x=x.unsqueeze(1);x=self.temporal_conv(x);x=self.spatial_conv(x);x=self.pool(x);x=self.projection(x);return x.squeeze(2).permute(0,2,1)

class MultiResolutionEncoder(nn.Module):
    def __init__(self,n_channels=3,n_samples=3000,embed_dim=128,dropout=0.1):
        super().__init__()
        self.enc_100=PatchEmbedding(n_channels,n_samples,embed_dim,dropout=dropout)
        self.enc_50=PatchEmbedding(n_channels,n_samples//2,embed_dim,dropout=dropout)
        self.enc_25=PatchEmbedding(n_channels,n_samples//4,embed_dim,dropout=dropout)
        self.merge=nn.Sequential(nn.Linear(embed_dim*3,embed_dim),nn.GELU(),nn.Dropout(dropout))
        self.seq_len_100=self.enc_100.seq_len
    def forward(self,x):
        e1=self.enc_100(x);e2=self.enc_50(x[:,:,::2]);e3=self.enc_25(x[:,:,::4]);T=e1.shape[1]
        e2=F.interpolate(e2.permute(0,2,1),size=T,mode='linear',align_corners=False).permute(0,2,1)
        e3=F.interpolate(e3.permute(0,2,1),size=T,mode='linear',align_corners=False).permute(0,2,1)
        return self.merge(torch.cat([e1,e2,e3],dim=-1))

class ContrastiveBoundaryModule(nn.Module):
    def __init__(self,embed_dim=128,hidden_dim=64,scales=(1,4,16),dropout=0.1):
        super().__init__()
        self.scales=scales
        self.projections=nn.ModuleList([nn.Sequential(nn.Linear(embed_dim,hidden_dim),nn.GELU(),nn.Dropout(dropout),nn.Linear(hidden_dim,hidden_dim)) for _ in scales])
        self.fusion=nn.Sequential(nn.Linear(len(scales),len(scales)*2),nn.GELU(),nn.Linear(len(scales)*2,1))
        self.temperature=nn.Parameter(torch.tensor(1.0))
    def _contrast(self,x,proj,offset):
        B,T,D=x.shape;h=F.normalize(proj(x),dim=-1)
        if offset<T:
            hs=torch.roll(h,-offset,dims=1);hs[:,-offset:,:]=h[:,-offset:,:]
            c=1.0-((h*hs).sum(dim=-1)+1.0)/2.0;c=torch.sigmoid((c-0.5)*self.temperature.abs().clamp(min=0.1))
        else:c=torch.zeros(B,T,device=x.device)
        return c
    def forward(self,x):
        ps=[self._contrast(x,p,o) for p,o in zip(self.projections,self.scales)]
        f=torch.sigmoid(self.fusion(torch.stack(ps,dim=-1)).squeeze(-1))
        return {'boundaries':f,'boundary_loss':0.01*sum(F.mse_loss(p,f.detach()) for p in ps)/len(ps)}

class OriginalRegimeMask(nn.Module):
    def __init__(self):super().__init__()
    def forward(self,b):c=torch.cumsum(b,dim=1);s=torch.exp(-torch.abs(c.unsqueeze(2)-c.unsqueeze(1)));return s,1-s

class RegimeStructuredAttention(nn.Module):
    def __init__(self,embed_dim=128,n_intra=4,n_inter=2,n_cross=2,dropout=0.1,regime_mask_module=None):
        super().__init__()
        self.n_heads=n_intra+n_inter+n_cross;self.head_dim=embed_dim//self.n_heads;self.n_intra=n_intra;self.n_inter=n_inter;self.scale=self.head_dim**-0.5
        self.qkv=nn.Linear(embed_dim,3*embed_dim);self.out_proj=nn.Linear(embed_dim,embed_dim)
        self.attn_drop=nn.Dropout(dropout);self.proj_drop=nn.Dropout(dropout);self.regime_mask=regime_mask_module or OriginalRegimeMask()
    def forward(self,x,boundaries,return_attention=False):
        B,T,D=x.shape;qkv=self.qkv(x).reshape(B,T,3,self.n_heads,self.head_dim).permute(2,0,3,1,4);q,k,v=qkv[0],qkv[1],qkv[2]
        attn=(q@k.transpose(-2,-1))*self.scale;s,c=self.regime_mask(boundaries);s=s.unsqueeze(1);c=c.unsqueeze(1)
        mask=torch.ones_like(attn);mask[:,:self.n_intra]=s.expand(B,self.n_intra,T,T);mask[:,self.n_intra:self.n_intra+self.n_inter]=c.expand(B,self.n_inter,T,T)
        attn=attn+torch.log(mask+1e-6);w=F.softmax(attn,dim=-1);w=self.attn_drop(w);out=(w@v).transpose(1,2).reshape(B,T,D);out=self.proj_drop(self.out_proj(out))
        if return_attention:return out,w
        return out

class NeuroStateBlock(nn.Module):
    def __init__(self,embed_dim=128,n_intra=4,n_inter=2,n_cross=2,mlp_ratio=4.0,dropout=0.1,regime_mask_module=None):
        super().__init__()
        self.norm1=nn.LayerNorm(embed_dim);self.attn=RegimeStructuredAttention(embed_dim,n_intra,n_inter,n_cross,dropout,regime_mask_module)
        self.norm2=nn.LayerNorm(embed_dim);self.mlp=nn.Sequential(nn.Linear(embed_dim,int(embed_dim*mlp_ratio)),nn.GELU(),nn.Dropout(dropout),nn.Linear(int(embed_dim*mlp_ratio),embed_dim),nn.Dropout(dropout))
    def forward(self,x,boundaries,return_attention=False):
        if return_attention:a,w=self.attn(self.norm1(x),boundaries,True);x=x+a;x=x+self.mlp(self.norm2(x));return x,w
        x=x+self.attn(self.norm1(x),boundaries);x=x+self.mlp(self.norm2(x));return x

class MultiResContrastiveNeuroState(nn.Module):
    def __init__(self,n_channels=3,n_samples=3000,n_classes=5,embed_dim=128,n_layers=4,dropout=0.1,contrast_scales=(1,4,16),cp_hidden=64,n_context_epochs=3):
        super().__init__()
        self.n_classes=n_classes;self.n_context=n_context_epochs
        self.mr_encoder=MultiResolutionEncoder(n_channels,n_samples,embed_dim,dropout)
        self.tokens_per_epoch=self.mr_encoder.seq_len_100;tt=self.tokens_per_epoch*n_context_epochs
        self.pos_embed=nn.Parameter(torch.randn(1,tt,embed_dim)*0.02);self.pos_drop=nn.Dropout(dropout)
        self.epoch_embed=nn.Parameter(torch.randn(1,n_context_epochs,1,embed_dim)*0.02)
        self.changepoint_module=ContrastiveBoundaryModule(embed_dim,cp_hidden,contrast_scales,dropout)
        self.blocks=nn.ModuleList([NeuroStateBlock(embed_dim,dropout=dropout) for _ in range(n_layers)])
        self.norm=nn.LayerNorm(embed_dim)
        self.head=nn.Sequential(nn.Linear(embed_dim,embed_dim//2),nn.GELU(),nn.Dropout(dropout),nn.Linear(embed_dim//2,n_classes))
        self.apply(self._iw)
    def _iw(self,m):
        if isinstance(m,nn.Linear):nn.init.trunc_normal_(m.weight,std=0.02)
        if hasattr(m,'bias') and m.bias is not None:nn.init.zeros_(m.bias)

In [8]:
# MAIN

def main():
    print("=" * 60)
    print("TCAV CONCEPT-BASED EXPLANATIONS")
    print("=" * 60)

    # --- Sleep-EDF ---
    sleep_h5 = Path("data/processed/sleep_edf_processed.h5")
    sleep_ckpt = Path("models/neurostate_acbl_isolated.pt")

    if sleep_h5.exists() and sleep_ckpt.exists():
        print("\n--- Loading Sleep-EDF model ---")
        model_sleep = MultiResContrastiveNeuroState(
            n_channels=3, n_samples=3000, n_classes=5,
            embed_dim=128, n_layers=4, dropout=0.1,
            n_context_epochs=3).to(device)
        ckpt = torch.load(sleep_ckpt, map_location=device, weights_only=False)
        state = ckpt['model_state_dict']
        fixed = {k.replace('enc_100hz','enc_100').replace('enc_50hz','enc_50').replace('enc_25hz','enc_25'): v for k,v in state.items()}
        model_sleep.load_state_dict(fixed, strict=False)
        print(f"  Loaded: {sleep_ckpt}")

        sleep_tcav = run_tcav_sleep(model_sleep, sleep_h5, device, figures_dir)

    # --- CHB-MIT ---
    chb_h5 = Path("data/processed/chbmit_combined_seizure_detection.h5")
    chb_ckpt = Path("models/acbl_isolation_chbmit_best.pt")

    if chb_h5.exists() and chb_ckpt.exists():
        print("\n--- Loading CHB-MIT model ---")
        model_chb = MultiResContrastiveNeuroState(
            n_channels=17, n_samples=3000, n_classes=2,
            embed_dim=128, n_layers=4, dropout=0.1,
            n_context_epochs=3).to(device)
        ckpt = torch.load(chb_ckpt, map_location=device, weights_only=False)
        model_chb.load_state_dict(ckpt['model_state_dict'])
        print(f"  Loaded: {chb_ckpt}")

        chb_tcav = run_tcav_seizure(model_chb, chb_h5, device, figures_dir)

    print("\nDone.")


if __name__ == "__main__":
    main()

TCAV CONCEPT-BASED EXPLANATIONS

--- Loading Sleep-EDF model ---
  Loaded: models/neurostate_acbl_isolated.pt

TCAV ANALYSIS — SLEEP-EDF
Computing frequency features for concept labeling...
Band powers: 100%|██████████| 7360/7360 [00:23<00:00, 315.07it/s]

  Concept: delta_dominant
    Wake: 6.7% positive
    N1: 3.4% positive
    N2: 33.6% positive
    N3: 67.9% positive
    REM: 3.5% positive

  Concept: spindle_present
    Wake: 54.8% positive
    N1: 50.5% positive
    N2: 19.8% positive
    N3: 2.9% positive
    REM: 28.9% positive

  Concept: alpha_dominant
    Wake: 59.8% positive
    N1: 53.7% positive
    N2: 12.7% positive
    N3: 1.5% positive
    REM: 28.2% positive
Extracting activations: 100%|██████████| 63/63 [00:00<00:00, 64.17it/s]

Training CAVs:
  delta_dominant: CAV accuracy = 0.909
  spindle_present: CAV accuracy = 0.821
  alpha_dominant: CAV accuracy = 0.868

TCAV Scores (>0.5 = concept positively influences prediction):
  Wake × delta_dominant: 0.560
  N1 × delta

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>